# Laboratory: Logical Reasoning for Planning
## Using an LLM to Construct and Test a Simple Planning Agent
**Course:** Undergraduate Artificial Intelligence  
**Core Thesis:** $\text{Logic} + \text{Search} = \text{Planning}$  

---

## 1. Learning Objectives
By the end of this laboratory, you will be able to:
1. Describe a planning problem using states, actions, preconditions, and effects (STRIPS formulation);
2. Determine whether an action is logically applicable in a given state ($S \models \text{Preconditions}(a)$);
3. Explain how logical reasoning and state-space search operate together in planning;
4. Use an LLM as an engineering tool to construct a propositional planning program in Python;
5. Systematically test and validate LLM-generated code across solvable, impossible, and edge-case scenarios;
6. Identify limitations of relying purely on an LLM to generate intelligent agents;
7. Explore formal logical verification using Prolog (Horn clauses, resolution inference).

---

## 2. Background: Formal Planning Foundations
A classical planning problem is defined as:
$$\mathcal{P} = (I, A, G)$$
where:
* $I$ is the **initial state** (a set of positive logical propositions);
* $A$ is the set of **available actions**;
* $G$ is the **goal specification** (a set of target propositions that must hold).

Each action $a \in A$ is defined by:
* **Preconditions:** Logical conditions that must hold in current state $S$ for action $a$ to be executable:
  $$S \models \text{Preconditions}(a)$$
* **Effects:** The state transformation rules:
  $$\text{Apply}(S, a) = (S \setminus \text{NegativeEffects}(a)) \cup \text{PositiveEffects}(a)$$


---
## 3. Motivating Scenario: Warehouse Package Delivery
A small warehouse robot operates in a warehouse with three connected locations:
$$\mathcal{L} = \{A, B, C\}$$
* The topology is a linear corridor: $A \leftrightarrow B \leftrightarrow C$.
* Initial configuration: The robot starts at location $A$, and a package is also at location $A$:
  $$I = \{\text{At}(\text{Robot}, A), \text{At}(\text{Package}, A)\}$$
* Goal configuration: The package must be delivered to location $C$:
  $$G = \{\text{At}(\text{Package}, C)\}$$

### Available Actions
1. **Move:** The robot can move between connected locations:
   * $\text{Move}(A, B)$, $\text{Move}(B, A)$, $\text{Move}(B, C)$, $\text{Move}(C, B)$.
2. **PickUp:** The robot picks up the package when both are at the same location.
3. **Drop:** The robot drops the package at its current location.


---
## Task 0: Understand the Planning Problem

### (a) Initial State $I$
$$I = \{\text{At}(\text{Robot}, A), \text{At}(\text{Package}, A)\}$$

### (b) Goal $G$
$$G = \{\text{At}(\text{Package}, C)\}$$

### (c) Available Actions
* $\text{Move}(A, B), \text{Move}(B, A), \text{Move}(B, C), \text{Move}(C, B)$
* $\text{PickUp}(\text{Package}, A), \text{PickUp}(\text{Package}, B), \text{PickUp}(\text{Package}, C)$
* $\text{Drop}(\text{Package}, A), \text{Drop}(\text{Package}, B), \text{Drop}(\text{Package}, C)$

### (d) Action Preconditions and Effects

| Action $a$ | Positive Preconditions | Negative Preconditions | Positive Effects (Add) | Negative Effects (Del) |
| :--- | :--- | :--- | :--- | :--- |
| $\text{Move}(X, Y)$ | $\text{At}(\text{Robot}, X)$ | - | $\text{At}(\text{Robot}, Y)$ | $\text{At}(\text{Robot}, X)$ |
| $\text{PickUp}(\text{Package}, X)$ | $\text{At}(\text{Robot}, X)$, $\text{At}(\text{Package}, X)$ | $\text{Holding}(\text{Package})$ | $\text{Holding}(\text{Package})$ | $\text{At}(\text{Package}, X)$ |
| $\text{Drop}(\text{Package}, X)$ | $\text{At}(\text{Robot}, X)$, $\text{Holding}(\text{Package})$ | - | $\text{At}(\text{Package}, X)$ | $\text{Holding}(\text{Package})$ |

---

### Questions

**Starting from $I = \{\text{At}(\text{Robot}, A), \text{At}(\text{Package}, A)\}$, is the action $\text{PickUp}(\text{Package}, A)$ applicable?**  
**Yes.** The preconditions for $\text{PickUp}(\text{Package}, A)$ are:
1. $\text{At}(\text{Robot}, A) \in I$ (Satisfied)
2. $\text{At}(\text{Package}, A) \in I$ (Satisfied)
3. $\text{Holding}(\text{Package}) \notin I$ (Satisfied)  
All preconditions are satisfied, so $I \models \text{Preconditions}(\text{PickUp}(\text{Package}, A))$.

**What about $\text{Drop}(\text{Package}, C)$?**  
**No.** The preconditions for $\text{Drop}(\text{Package}, C)$ are:
1. $\text{At}(\text{Robot}, C)$
2. $\text{Holding}(\text{Package})$  
Neither condition is present in state $I$. Hence, $I \not\models \text{Preconditions}(\text{Drop}(\text{Package}, C))$, and the action is logically inapplicable.


---
## Task 1: Construct a Plan by Hand
Before implementing the software planner, we manually construct the step-by-step state transition trajectory:

$$S_0 \xrightarrow{a_1} S_1 \xrightarrow{a_2} S_2 \xrightarrow{a_3} S_3 \xrightarrow{a_4} S_4$$

### Step-by-Step State Trace

| State | Action Applied | Resulting State Facts | Goal Satisfied? |
| :--- | :--- | :--- | :--- |
| **$S_0$** | *(Initial State)* | $\{\text{At}(\text{Robot}, A), \text{At}(\text{Package}, A)\}$ | False |
| **$S_1$** | $\text{PickUp}(\text{Package}, A)$ | $\{\text{At}(\text{Robot}, A), \text{Holding}(\text{Package})\}$ | False |
| **$S_2$** | $\text{Move}(A, B)$ | $\{\text{At}(\text{Robot}, B), \text{Holding}(\text{Package})\}$ | False |
| **$S_3$** | $\text{Move}(B, C)$ | $\{\text{At}(\text{Robot}, C), \text{Holding}(\text{Package})\}$ | False |
| **$S_4$** | $\text{Drop}(\text{Package}, C)$ | $\{\text{At}(\text{Robot}, C), \text{At}(\text{Package}, C)\}$ | **True** ($S_4 \models G$) |


---
## Task 2: Implementation of the Logical Planning Agent
We implement a propositional planning agent in Python using Breadth-First Search (BFS) over the state space.
Each action specifies positive/negative preconditions and positive/negative effects.


In [1]:
"""
Logical Planning Agent Framework (STRIPS-style)
Uses Breadth-First Search (BFS) over propositional states.
"""
from collections import deque
from typing import Set, List, Dict, Optional, FrozenSet

class Action:
    def __init__(self, name: str, 
                 pos_pre: Set[str], neg_pre: Set[str], 
                 pos_eff: Set[str], neg_eff: Set[str]):
        self.name = name
        self.pos_pre = set(pos_pre)
        self.neg_pre = set(neg_pre)
        self.pos_eff = set(pos_eff)
        self.neg_eff = set(neg_eff)

    def is_applicable(self, state: Set[str]) -> bool:
        """Action is applicable if all pos_pre are present and no neg_pre are present."""
        return self.pos_pre.issubset(state) and self.neg_pre.isdisjoint(state)

    def apply(self, state: Set[str]) -> Set[str]:
        """Apply action: remove negative effects, add positive effects."""
        return (state - self.neg_eff) | self.pos_eff

    def __repr__(self):
        return self.name


class LogicalPlanner:
    def __init__(self, actions: List[Action]):
        self.actions = actions
        self.states_expanded = 0

    def plan(self, initial_state: Set[str], goal_state: Set[str]) -> Optional[Dict]:
        """
        Finds a plan from initial_state to goal_state using BFS.
        Returns dictionary with plan sequence and intermediate states.
        """
        init_frozen = frozenset(initial_state)
        goal_set = set(goal_state)

        # BFS queue stores: (current_state_frozen, action_history, state_history)
        frontier = deque([(init_frozen, [], [set(initial_state)])])
        visited: Set[FrozenSet[str]] = {init_frozen}
        self.states_expanded = 0

        while frontier:
            current_frozen, actions_taken, states_visited = frontier.popleft()
            self.states_expanded += 1
            current_set = set(current_frozen)

            # Goal Test: Goal propositions must be subset of current state
            if goal_set.issubset(current_set):
                return {
                    "success": True,
                    "plan": actions_taken,
                    "states": states_visited,
                    "states_expanded": self.states_expanded
                }

            # Explore applicable actions
            for action in self.actions:
                if action.is_applicable(current_set):
                    next_state = action.apply(current_set)
                    next_frozen = frozenset(next_state)

                    if next_frozen not in visited:
                        visited.add(next_frozen)
                        frontier.append((
                            next_frozen,
                            actions_taken + [action.name],
                            states_visited + [next_state]
                        ))

        return {
            "success": False,
            "plan": [],
            "states": [],
            "states_expanded": self.states_expanded
        }

print("LogicalPlanner class successfully defined.")


LogicalPlanner class successfully defined.


---
## Task 3: Test the Generated Planner
We test the planner across three distinct scenarios:
* **Test A: Solvable Problem** (Standard warehouse problem)
* **Test B: Impossible Problem** (Removing `PickUp` action; planner must report "No plan found")
* **Test C: Irrelevant Actions** (Adding robot movements without the package; planner must not confuse robot reaching C with package reaching C)


In [1]:
def create_warehouse_actions(include_pickup=True) -> List[Action]:
    actions = [
        # Move actions
        Action("Move(A, B)", {"At(Robot, A)"}, set(), {"At(Robot, B)"}, {"At(Robot, A)"}),
        Action("Move(B, A)", {"At(Robot, B)"}, set(), {"At(Robot, A)"}, {"At(Robot, B)"}),
        Action("Move(B, C)", {"At(Robot, B)"}, set(), {"At(Robot, C)"}, {"At(Robot, B)"}),
        Action("Move(C, B)", {"At(Robot, C)"}, set(), {"At(Robot, B)"}, {"At(Robot, C)"}),
        
        # Drop actions
        Action("Drop(Package, A)", {"At(Robot, A)", "Holding(Package)"}, set(), 
               {"At(Package, A)"}, {"Holding(Package)"}),
        Action("Drop(Package, B)", {"At(Robot, B)", "Holding(Package)"}, set(), 
               {"At(Package, B)"}, {"Holding(Package)"}),
        Action("Drop(Package, C)", {"At(Robot, C)", "Holding(Package)"}, set(), 
               {"At(Package, C)"}, {"Holding(Package)"}),
    ]
    if include_pickup:
        actions.extend([
            Action("PickUp(Package, A)", {"At(Robot, A)", "At(Package, A)"}, {"Holding(Package)"}, 
                   {"Holding(Package)"}, {"At(Package, A)"}),
            Action("PickUp(Package, B)", {"At(Robot, B)", "At(Package, B)"}, {"Holding(Package)"}, 
                   {"Holding(Package)"}, {"At(Package, B)"}),
            Action("PickUp(Package, C)", {"At(Robot, C)", "At(Package, C)"}, {"Holding(Package)"}, 
                   {"Holding(Package)"}, {"At(Package, C)"}),
        ])
    return actions

print("=================== TEST A: SOLVABLE WAREHOUSE PROBLEM ===================")
initial_state = {"At(Robot, A)", "At(Package, A)"}
goal_state = {"At(Package, C)"}
planner_a = LogicalPlanner(create_warehouse_actions(include_pickup=True))
res_a = planner_a.plan(initial_state, goal_state)

print(f"Plan Found: {res_a['success']}")
print(f"States Expanded: {res_a['states_expanded']}")
print(f"Plan Length: {len(res_a['plan'])}")
print("\nExecution Trace:")
for step_idx, (act, st) in enumerate(zip(["START"] + res_a['plan'], res_a['states'])):
    print(f"  Step {step_idx}: Action = {act}")
    print(f"          State  = {sorted(st)}")

print("\n=================== TEST B: IMPOSSIBLE PROBLEM ===================")
# Remove PickUp action
planner_b = LogicalPlanner(create_warehouse_actions(include_pickup=False))
res_b = planner_b.plan(initial_state, goal_state)
print(f"Plan Found: {res_b['success']} (Expected: False -> 'No plan found')")
print(f"States Expanded: {res_b['states_expanded']}")
if not res_b['success']:
    print("Result: Correctly reported 'No plan found' without hallucinating an action.")

print("\n=================== TEST C: IRRELEVANT ACTIONS ===================")
# Robot starts at A, Package at A. Goal is At(Package, C). 
# Robot moves back and forth to C without picking up package.
res_c = planner_a.plan(initial_state, goal_state)
robot_at_c = {"At(Robot, C)"}
print(f"Testing whether planner confuses Robot reaching C with Package reaching C:")
print(f"Robot reaching C satisfied? {robot_at_c.issubset(res_c['states'][-1])}")
print(f"Package reaching C satisfied? {goal_state.issubset(res_c['states'][-1])}")
print("Verified: Goal predicate explicitly checked At(Package, C).")


=================== TEST A: SOLVABLE WAREHOUSE PROBLEM ===================
Plan Found: True
States Expanded: 8
Plan Length: 4

Execution Trace:
  Step 0: Action = START
          State  = ['At(Package, A)', 'At(Robot, A)']
  Step 1: Action = PickUp(Package, A)
          State  = ['At(Robot, A)', 'Holding(Package)']
  Step 2: Action = Move(A, B)
          State  = ['At(Robot, B)', 'Holding(Package)']
  Step 3: Action = Move(B, C)
          State  = ['At(Robot, C)', 'Holding(Package)']
  Step 4: Action = Drop(Package, C)
          State  = ['At(Package, C)', 'At(Robot, C)']

=================== TEST B: IMPOSSIBLE PROBLEM ===================
Plan Found: False (Expected: False -> 'No plan found')
States Expanded: 3
Result: Correctly reported 'No plan found' without hallucinating an action.

=================== TEST C: IRRELEVANT ACTIONS ===================
Testing whether planner confuses Robot reaching C with Package reaching C:
Robot reaching C satisfied? True
Package reaching C satisfie

---
## Task 4: Logic and Search
A planning agent fundamentally combines two distinct computational mechanisms:
* **Logical Reasoning:** Operates at the **local / operator level**. It checks whether the current state logically entails the preconditions of candidate actions ($S \models \text{Preconditions}(a)$) and computes the exact logical consequence of applying an action ($S' = \text{Apply}(S, a)$).
* **Search:** Operates at the **global / path level**. It systematically explores sequences of state transitions across alternative branches to find a path from $I$ to $G$.

### Completed Description Diagram
```text
      Current state (S)
              |
              v
  Check action preconditions (Logic: S |= Preconditions(a))
              |
              v
  [ Filter Applicable Actions ]  <--- (Logical Filter)
              |
              v
    Generate successor state (Logic: Apply(S, a))
              |
              v
   Search over alternatives (Search: Frontier / BFS / A*)
              |
              v
            Goal? (Logic: S_n |= G)
```

> **Think About It:** *"Logic determines what is possible; search determines what to try."*  
> Logic acts as the physics/rules engine validating which moves can occur, while search provides the strategy exploring candidate trajectories until a goal condition is achieved.


---
## Task 5: Can the LLM Verify Its Own Plan?

### LLM Plan Explanation vs Independent Execution
When an LLM is asked to verify a plan, it generates an explanation in natural language:
> *"At step 1, the robot is at A and the package is at A, so PickUp is valid. Then the robot moves to B, then to C, and drops the package at C. This satisfies the goal."*

### Evaluation Question: Which should you trust more?
**(a) The LLM's explanation;**  
**(b) The independently executed state transitions?**

**Answer: (b) The independently executed state transitions.**

### Rigorous Explanation:
1. **Verifiable Soundness vs Probabilistic Generation:** An independent execution engine evaluates exact boolean set memberships ($S \models \text{Preconditions}(a)$). It cannot hallucinate missing facts or overlook unsatisfied preconditions.
2. **LLM Confirmation Bias & Fluency:** LLMs are trained to generate coherent, persuasive text. An LLM can readily produce an elegant, seemingly convincing narrative explaining why an invalid plan works (e.g., claiming a robot dropped a package it never picked up).
3. **Engineering Principle:**  
   $$\mathbf{\text{A generated explanation is not the same as an independent verification.}}$$


---
## Section 5: Reflection Questions

### 1. Why is it useful to specify action preconditions and effects before asking an LLM to write the planner?
Specifying preconditions and effects provides a formal, unambiguous specification (STRIPS semantics). Without this, LLMs make implicit, inconsistent domain assumptions (such as assuming the package teleports with the robot without a `PickUp` action, or allowing picking up objects remotely).

### 2. Give an example of an error that could occur if the planner failed to check an action's preconditions.
If the planner omitted checking $\text{Holding}(\text{Package})$ before executing $\text{Drop}(\text{Package}, C)$, the robot could simply move to $C$ while empty-handed and execute `Drop`, magically materializing the package at $C$.

### 3. Why is a plan that "looks reasonable" not necessarily a valid plan?
A plan might appear intuitive to human inspection (e.g., *Move to warehouse, load box, move to dispatch, unload*), but violate subtle temporal or causal constraints—such as trying to open a door before unlocking it, moving while unladen, or exceeding payload limits. Only formal logical deduction against every intermediate state guarantees validity.

### 4. What did the LLM contribute to the implementation?
The LLM rapidly generated the Python class structure, set arithmetic operations (`issubset`, `isdisjoint`, set difference/union), and BFS queue skeleton, drastically reducing boilerplate coding time.

### 5. What did you have to verify independently?
We independently verified:
* Correct STRIPS delete-before-add update semantics;
* Accurate termination when no plan exists (preventing infinite loops);
* Prevention of state hash mutability errors (converting Python sets to `frozenset` for visited set membership).

### 6. In this laboratory, where is logical reasoning being used?
Logical reasoning is used in:
1. Evaluating action preconditions: $S \models \text{Preconditions}(a)$;
2. Computing successor states via negative/positive effect propagation;
3. Evaluating the goal predicate: $S_n \models G$.

### 7. How is planning related to the search algorithms studied in the previous module?
Planning **is** state-space search, but over states described by structured logical propositions rather than simple atomic coordinates. The transition function $T(s, a)$ is dynamically generated by logical theorem proving / precondition satisfaction rather than an explicit adjacency matrix.


---
## Section 7: Optional Extension – Prolog as a Logical Verifier
Prolog is a declarative logic programming language based on Horn-clause first-order logic:
$$\text{Facts} + \text{Rules} + \text{Inference} = \text{Query Answer}$$

We demonstrate how a logical knowledge base written in Prolog can independently verify plans generated by a Python agent.
The facts and rules from `planner.pl`:
```prolog
connected(a, b).
connected(b, a).
connected(b, c).
connected(c, b).

can_move(X, Y) :- connected(X, Y).
valid_move(X, Y) :- connected(X, Y).
```


In [1]:
"""
Declarative Logic Verifier (Prolog-style resolution in Python)
Simulates Horn-Clause Resolution for Warehouse Verification
"""
class PrologKnowledgeBase:
    def __init__(self):
        self.facts = set()
        self.rules = []

    def add_fact(self, predicate: str, *args):
        self.facts.add((predicate, tuple(args)))

    def query(self, predicate: str, *args) -> bool:
        # Direct fact check
        if (predicate, tuple(args)) in self.facts:
            return True
        # Rule: can_move(X, Y) :- connected(X, Y)
        if predicate in ("can_move", "valid_move") and len(args) == 2:
            return ("connected", (args[0], args[1])) in self.facts
        # Task 8 rules
        if predicate == "slippery":
            return self.query("wet_road")
        if predicate == "reduce_speed":
            return self.query("slippery")
        return False

# Initialize Knowledge Base
kb = PrologKnowledgeBase()
kb.add_fact("connected", "a", "b")
kb.add_fact("connected", "b", "a")
kb.add_fact("connected", "b", "c")
kb.add_fact("connected", "c", "b")

print("=================== TASK 6: PROLOG QUERY VERIFICATION ===================")
q1 = kb.query("can_move", "a", "b")
q2 = kb.query("can_move", "a", "c")
print(f"?- can_move(a, b).  -->  {q1} (Expected: True)")
print(f"?- can_move(a, c).  -->  {q2} (Expected: False)")

print("\n=================== TASK 7: CHECKING A PROPOSED PLAN ===================")
plan_moves = [("a", "b"), ("b", "c")]
print("Checking proposed sequence: Move(a, b), Move(b, c)")
all_valid = True
for u, v in plan_moves:
    is_valid = kb.query("valid_move", u, v)
    print(f"  ?- valid_move({u}, {v}). --> {is_valid}")
    if not is_valid:
        all_valid = False
print(f"Overall Plan Valid: {all_valid}")

print("\nChallenge: Suppose Python planner proposes Move(a, c):")
direct_invalid = kb.query("valid_move", "a", "c")
print(f"  ?- valid_move(a, c). --> {direct_invalid}")
print("Prolog independently rejects Move(a, c) as unprovable from the knowledge base.")

print("\n=================== TASK 8: LOGICAL DEDUCTION CHAIN ===================")
kb.add_fact("wet_road")
q_speed = kb.query("reduce_speed")
print(f"?- reduce_speed. --> {q_speed}")
print("Deduction Sequence: wet_road => slippery => reduce_speed")


=================== TASK 6: PROLOG QUERY VERIFICATION ===================
?- can_move(a, b).  -->  True (Expected: True)
?- can_move(a, c).  -->  False (Expected: False)

=================== TASK 7: CHECKING A PROPOSED PLAN ===================
Checking proposed sequence: Move(a, b), Move(b, c)
  ?- valid_move(a, b). --> True
  ?- valid_move(b, c). --> True
Overall Plan Valid: True

Challenge: Suppose Python planner proposes Move(a, c):
  ?- valid_move(a, c). --> False
Prolog independently rejects Move(a, c) as unprovable from the knowledge base.

=================== TASK 8: LOGICAL DEDUCTION CHAIN ===================
?- reduce_speed. --> True
Deduction Sequence: wet_road => slippery => reduce_speed


---
### Analysis of Tasks 6, 7, 8 and Reflection 7.2

#### Task 6 Questions
**(a) Why does Prolog return true for `can_move(a,b)`?**  
Because `can_move(a,b)` resolves via the rule `can_move(X,Y) :- connected(X,Y)` to the sub-goal `connected(a,b)`. The ground fact `connected(a,b).` is explicitly present in the knowledge base, satisfying the query.

**(b) Why does it not establish `can_move(a,c)`?**  
Under Prolog's **Closed-World Assumption (CWA)** and negation-as-failure, any proposition that cannot be deduced from facts and rules is deemed false. There is no fact `connected(a,c)` and no direct path rule connecting `a` to `c` in a single step.

**(c) What is the relationship between the Prolog rule and logical implication?**  
The Prolog rule `can_move(X, Y) :- connected(X, Y).` is syntactically a Definite Horn Clause:
$$\text{Connected}(X, Y) \to \text{CanMove}(X, Y)$$
or in clausal normal form: $\neg \text{Connected}(X, Y) \lor \text{CanMove}(X, Y)$.

---

#### Task 8: Logical Reasoning Sequence
Consider:
1. `wet_road.` (Fact)
2. `slippery :- wet_road.` (Rule: $\text{WetRoad} \to \text{Slippery}$)
3. `reduce_speed :- slippery.` (Rule: $\text{Slippery} \to \text{ReduceSpeed}$)

Query: `?- reduce_speed.`  
Logical reasoning chain:
$$\mathbf{\text{Fact (wet\_road)} \implies \text{Rule (wet\_road } \to \text{ slippery)} \implies \text{Rule (slippery } \to \text{ reduce\_speed)} \implies \text{Conclusion (reduce\_speed)}}$$

---

#### Section 7.2 Reflection Questions
1. **What is the difference between a Prolog fact and a Prolog rule?**  
   A **fact** asserts an unconditional truth about the domain (an axiom with empty body, e.g., $P$). A **rule** asserts a conditional truth that holds if its body sub-goals can be satisfied (an implication $B_1 \land B_2 \dots \to H$).

2. **How does a Prolog query correspond to asking whether something follows from a knowledge base?**  
   A query `?- Q` poses whether the knowledge base $\text{KB}$ logically entails $Q$ ($\text{KB} \models Q$). Prolog attempts to prove this by refutation: it adds $\neg Q$ to $\text{KB}$ and uses SLD resolution to derive a contradiction (the empty clause $\Box$).

3. **Why might it be useful to use a Prolog program to verify a plan generated by a Python program?**  
   It decouples the *generator* from the *verifier*. Python might use heuristics, neural networks, or LLMs to search for candidate plans rapidly, while Prolog serves as a clean, formally verifiable, declarative specification of valid domain transitions.

4. **What advantage does an independent verifier provide when the original plan was generated with the help of an LLM?**  
   It creates a **neuro-symbolic guardrail**:
   $$\mathbf{\text{Generate (LLM / Heuristic)} \longrightarrow \text{Independent Logical Verifier (Prolog)}}$$
   The LLM proposes creative or complex plans, but the logical verifier independently and deterministically guarantees safety, physical feasibility, and rule compliance before actions are executed in the real world.
